# TCP/Reno и очередь RED

**Цель:** исследовать окно перегрузки TCP/Reno и состояние очереди RED в
многопоточной сети с узким местом 20 Мбит/с.

In [1]:
using DrWatson
@quickactivate "project"
ENV["GKSwstype"] = "100"
using CSV, DataFrames, Plots, Statistics
include(srcdir("tcp_red.jl"))
using .TCPRED

name = "01_tcp_red"
mkpath(datadir(name)); mkpath(plotsdir(name))
network = NetworkParameters(flows=25, capacity_bps=20e6, packet_bytes=500,
    propagation_rtt=0.118, duration=30.0, dt=0.002)
red = REDParameters(qmin=75.0, qmax=150.0, weight=0.002, pmax=0.1, limit=300.0)
result = simulate_tcp_red(; network, red, seed=202604)

trajectory = DataFrame(time=result.time, cwnd_first=result.cwnd_first,
    cwnd_mean=result.cwnd_mean, queue=result.queue,
    average_queue=result.average_queue, drop_probability=result.drop_probability,
    throughput_mbps=result.throughput_mbps)
CSV.write(datadir(name, "trajectory.csv"), trajectory)

warm = trajectory.time .>= 10.0
summary = DataFrame(metric=["mean_cwnd", "mean_queue", "mean_average_queue",
    "mean_throughput_mbps", "loss_events"],
    value=[mean(trajectory.cwnd_mean[warm]), mean(trajectory.queue[warm]),
        mean(trajectory.average_queue[warm]), mean(trajectory.throughput_mbps[warm]),
        result.losses])
CSV.write(datadir(name, "summary.csv"), summary)
println("=== TCP/Reno + RED: базовый эксперимент ===")
println("Потоков: $(network.flows), узкое место: $(network.capacity_bps/1e6) Мбит/с")
show(summary; allrows=true, allcols=true); println()

default(fontfamily="DejaVu Sans", linewidth=2.2, framestyle=:box, gridalpha=0.22)
p1 = plot(trajectory.time, trajectory.cwnd_first; label="первый поток",
    xlabel="Время, с", ylabel="cwnd, пакеты", title="Окно перегрузки TCP/Reno",
    size=(1000,620), color=:blue)
plot!(p1, trajectory.time, trajectory.cwnd_mean; label="среднее по потокам", color=:black)
savefig(p1, plotsdir(name, "cwnd.png"))

p2 = plot(trajectory.time, trajectory.queue; label="мгновенная очередь",
    xlabel="Время, с", ylabel="Пакеты", title="Очередь RED на узком месте",
    size=(1000,620), color=:orange)
plot!(p2, trajectory.time, trajectory.average_queue; label="EWMA", color=:red)
hline!(p2, [red.qmin, red.qmax]; label=["qmin" "qmax"], linestyle=:dash)
savefig(p2, plotsdir(name, "red_queue.png"))

p3 = plot(trajectory.time, trajectory.drop_probability; label="p(t)",
    xlabel="Время, с", ylabel="Вероятность", title="Вероятность раннего сброса RED",
    size=(1000,620), color=:purple)
savefig(p3, plotsdir(name, "drop_probability.png"))

p4 = plot(trajectory.time, trajectory.throughput_mbps; label="пропускная способность",
    xlabel="Время, с", ylabel="Мбит/с", title="Использование узкого места",
    size=(1000,620), color=:green)
hline!(p4, [network.capacity_bps/1e6]; label="20 Мбит/с", linestyle=:dash, color=:black)
savefig(p4, plotsdir(name, "throughput.png"))

=== TCP/Reno + RED: базовый эксперимент ===


Потоков: 25, узкое место: 20.0 Мбит/с
5×2 DataFrame
 Row │ metric                value    
     │ String                Float64  
─────┼────────────────────────────────
   1 │ mean_cwnd              24.5234
   2 │ mean_queue             58.7519
   3 │ mean_average_queue     56.4558
   4 │ mean_throughput_mbps   18.7654
   5 │ loss_events           446.0


"/workspace/labs/lab04/project/plots/01_tcp_red/throughput.png"

RED реагирует на сглаженную длину очереди, а TCP/Reno отвечает на потери
мультипликативным уменьшением окна, формируя характерную пилообразную кривую.